# 03. PyRPMD 固体与周期性材料计算模块案例 (Solid-State & Periodic Materials)

本教程演示 **PyRPMD** 在固体材料、表面吸附与周期性凝聚相体系中的四大核心计算工作流：

1. **周期性晶胞平衡路径积分分子动力学 (`task = "pimd"`)**：在三维晶胞与周期性边界条件（`cell` & `pbc`）下运行平衡 PIMD，提取原初动能估计量 $K_{\mathrm{prim}}$、质心维里动能估计量 $K_{\mathrm{cv}}$ 以及表征原子核量子离域程度的环聚合物回转半径平方 $R_g^2$。
2. **固定基底原子的表面板层模型 (`fixed_atoms` Slab PIMD)**：模拟重金属基底固定（`fixed_atoms = [0]`）而轻吸附原子（如 $\mathrm{H}$）在表面周期势场中发生量子零点涨落。
3. **各向同性恒压系综 (`barostat = "bzp"` NPT PIMD)**：使用 BZP 随机恒压器模拟有限压强（如 $1\,\mathrm{GPa}$）下的晶胞体积涨落与焓/热力学估计量。
4. **周期性固体质心扩散 (`task = "diffusion"`) 与晶格间隙位点逃逸速率 (`task = "site_rate"`)**：
   - 使用未包裹（unwrapped）质心轨迹计算均方位移（MSD）、速度自相关函数（VACF）并自动识别弹道运动与扩散区；
   - 沿分数坐标集体变量（`kind = "fractional"`）进行周期势垒伞形抽样、盆地归一化与分割面回穿采样，获得固体中氢扩散/位点跃迁的量子速率 $k_{\mathrm{RPMD}}$。

In [ ]:
from pathlib import Path
import tempfile
import json
import numpy as np

import pyrpmd
from pyrpmd import JobSpec, System
from pyrpmd.runner import run, resume_run

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    ROOT = Path(pyrpmd.__file__).resolve().parents[2]

print(f"PyRPMD version: {pyrpmd.__version__}")
print(f"Materials capabilities: {json.dumps(pyrpmd.runner.classify_stability({'task': 'pimd'}), indent=2)}")

## 1. 周期性固体平衡 PIMD：量子能量估计量与核量子离域 ($R_g^2$)

在周期性体系中，核量子效应（NQE）使轻原子（如 $\mathrm{H}$、$\mathrm{D}$、$\mathrm{Li}$）在晶格位点附近发生显著的空间离域。
- **原初动能估计量（Primitive Estimator）**：
  $$K_{\mathrm{prim}} = \frac{3NP}{2\beta} - \frac{1}{P}\sum_{j=1}^{P} \frac{1}{2} m \omega_P^2 (\mathbf{q}_j - \mathbf{q}_{j+1})^2$$
- **质心维里动能估计量（Centroid-Virial Estimator，方差不随 $P$ 增大而发散）**：
  $$K_{\mathrm{cv}} = \frac{3N}{2\beta} - \frac{1}{2P}\sum_{j=1}^{P} (\mathbf{q}_j - \bar{\mathbf{q}}) \cdot \mathbf{F}(\mathbf{q}_j)$$

下面我们对比经典极限（$P=1$）与量子环聚合物（$P=8$）在周期余弦晶格势 `periodic_cosine` 中的能量与回转半径平方 $R_g^2$。

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)

    comparison = {}
    for beads in (1, 8):
        spec = JobSpec.from_file(ROOT / "examples/materials/pimd.inp")
        spec.data["beads"] = beads
        spec.data["sampling"].update(
            equilibration="4 fs", production="16 fs", interval="0.5 fs", replicas=2
        )
        out_dir = tmp_path / f"pimd_P{beads}"
        run(spec, output=out_dir)
        summary = json.loads((out_dir / "results/summary.json").read_text())
        comparison[f"P={beads}"] = {
            "potential_energy_hartree": summary["estimators"]["potential_energy"]["mean"],
            "kinetic_cv_hartree": summary["estimators"]["kinetic_centroid_virial"]["mean"],
            "internal_energy_cv_hartree": summary["estimators"]["internal_energy_centroid_virial"]["mean"],
            "radius_of_gyration_squared_bohr2": float(np.mean(summary["radius_of_gyration_squared_bohr2"])),
        }

print(json.dumps(comparison, indent=2))
# P=1 经典极限下环聚合物回转半径严格为 0，而 P=8 呈现显著量子离域
assert comparison["P=1"]["radius_of_gyration_squared_bohr2"] == 0.0
assert comparison["P=8"]["radius_of_gyration_squared_bohr2"] > 0.01

## 2. 固定基底原子的表面板层模型 (`fixed_slab.inp`) 与各向同性恒压 NPT (`npt_free.inp`)

- **板层模型（Fixed-Atom Slab）**：通过 `system.fixed_atoms = [0]` 将基底原子（如 `Ni`）固定在晶格坐标上，仅对可移动吸附原子（如 `H`）传播环聚合物并累计量子动能自由度（$3 N_{\mathrm{mobile}}$）。
- **各向同性 BZP 恒压器（Isotropic NPT）**：在三维全周期性边界条件（`pbc = [true, true, true]`）下耦合晶胞体积与质心坐标，支持外压控制。

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)

    # 2.1 固定基底 Slab PIMD
    slab_spec = JobSpec.from_file(ROOT / "examples/materials/fixed_slab.inp")
    slab_spec.data["sampling"].update(
        equilibration_steps=20, production_steps=60, sample_interval=5, replicas=2
    )
    run(slab_spec, output=tmp_path / "fixed_slab")
    slab_summary = json.loads((tmp_path / "fixed_slab/results/summary.json").read_text())

    # 2.2 各向同性 BZP NPT PIMD
    npt_spec = JobSpec.from_file(ROOT / "examples/materials/npt_free.inp")
    npt_spec.data["sampling"].update(
        equilibration="4 fs", production="16 fs", interval="1 fs", replicas=2
    )
    run(npt_spec, output=tmp_path / "npt")
    npt_summary = json.loads((tmp_path / "npt/results/summary.json").read_text())

print(json.dumps({
    "fixed_slab": {
        "ensemble": slab_summary["ensemble"],
        "fixed_atoms": slab_summary["fixed_atoms"],
        "mobile_atoms": slab_summary["mobile_atoms"],
        "internal_energy_cv_hartree": slab_summary["estimators"]["internal_energy_centroid_virial"]["mean"],
    },
    "isotropic_npt": {
        "ensemble": npt_summary["ensemble"],
        "kinetic_cv_hartree": npt_summary["estimators"]["kinetic_centroid_virial"]["mean"],
        "samples_per_replica": npt_summary["samples_per_replica"],
    },
}, indent=2))

assert slab_summary["fixed_atoms"] == [0]
assert slab_summary["mobile_atoms"] == 1
assert npt_summary["ensemble"] == "npt" 

## 3. 固体质心输运 (`diffusion`) 与晶格位点跃迁速率 (`site_rate`)

在周期性固体中研究间隙原子（如金属中的氢同位素）迁移有两种互补途径：
1. **高温/低垒直接输运 (`task = "diffusion"`)**：先在 PILE-L 热浴下平衡环聚合物，再切换至无热浴的物理质量 RPMD 轨迹，记录跨晶胞边界展开（unwrapped）的质心位移。PyRPMD 会自动检验对数斜率，若轨迹仍处于弹道区（如自由粒子对照算例 `diffusion.inp`），会诚实拒绝报告伪扩散系数并标记 `ballistic_or_drifting`。
2. **稀有事件位点跃迁速率 (`task = "site_rate"`)**：沿晶胞分数坐标（`kind = "fractional"`）定义反应盆地 `basin = [-0.5, 0.5]` 与分割面 `dividing_surface = 0.5`，结合伞形采样 PMF 与分割面回穿给出单向逃逸速率 $k_{\mathrm{site}}(T)$（单位：$\mathrm{s^{-1}}$）。

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)

    # 3.1 运行周期自由粒子扩散负对照算例，验证弹道运动诊断门禁
    diff_spec = JobSpec.from_file(ROOT / "examples/materials/diffusion.inp")
    diff_spec.data["sampling"].update(
        equilibration="4 fs", production="20 fs", interval="0.5 fs", replicas=2
    )
    diff_spec.data["diffusion"].update(max_lag=20, fit_range=["2 fs", "8 fs"], bootstrap=10)
    run(diff_spec, output=tmp_path / "diffusion")
    diff_summary = json.loads((tmp_path / "diffusion/results/summary.json").read_text())

    # 3.2 运行周期晶格位点逃逸速率 (site_rate)
    site_spec = JobSpec.from_file(ROOT / "examples/materials/site_rate.inp")
    site_spec.data["sampling"].update(
        equilibration="5 fs", production="20 fs", interval="0.5 fs", replicas=2
    )
    site_spec.data["analysis"].update(bins=12, min_count=1, bootstrap=0)
    site_spec.data["recrossing"].update(
        parent_replicas=2,
        parent_samples=2,
        equilibration="2 fs",
        parent_interval="1 fs",
        child_pairs=1,
        child_time="4 fs",
        bootstrap=0,
    )
    run(site_spec, output=tmp_path / "site_rate")
    site_summary = json.loads((tmp_path / "site_rate/results/summary.json").read_text())

print(json.dumps({
    "diffusion_diagnostic": {
        "task": diff_summary["task"],
        "status": diff_summary["status"],
        "diffusion_coefficient_reported": diff_summary.get("diffusion_coefficient_cm2_s"),
    },
    "periodic_site_rate": {
        "task": site_summary["task"],
        "k_qtst_s_inv": site_summary["k_qtst_s_inv"],
        "rate_estimate_s_inv": site_summary["rate_estimate_s_inv"],
        "propagation_plan": site_summary["propagation_plan"],
    },
}, indent=2))

assert diff_summary.get("diffusion_coefficient_cm2_s") is None
assert site_summary["k_qtst_s_inv"] > 0.0
assert site_summary["rate_estimate_s_inv"] > 0.0